# Face Recognition API Testing Notebook

This notebook tests the Face Recognition Auto-Update Integration (SO-124)

## Setup

In [ ]:
import requests
import json

# Configuration
FR_API_URL = "http://localhost:8000"  # Change if needed
BACKEND_URL = "http://localhost:3000"  # Change if needed
CLIENT_SLUG = "org_humblebee"  # Your client slug

## Test 1: Health Check

In [ ]:
response = requests.get(f"{FR_API_URL}/api/v1/health")
print(f"Status Code: {response.status_code}")
print(f"Response: {json.dumps(response.json(), indent=2)}")

## Test 2: Add User Embedding (Simulate Backend Notification)

In [ ]:
# Simulate adding a user
payload = {
    "action": "add_user",
    "client_slug": CLIENT_SLUG,
    "user_data": {
        "id": 999,
        "full_name": "Test User",
        "external_id": "TEST001",
        "image_urls": [
            # Add real GCS URLs here if you have them
            "https://storage.googleapis.com/your-bucket/test-user.jpg"
        ]
    }
}

response = requests.post(
    f"{FR_API_URL}/api/v1/embeddings/update",
    json=payload,
    headers={"Content-Type": "application/json"}
)

print(f"Status Code: {response.status_code}")
print(f"Response: {json.dumps(response.json(), indent=2)}")

## Test 3: Update User Embedding

In [ ]:
payload = {
    "action": "update_user",
    "client_slug": CLIENT_SLUG,
    "user_data": {
        "id": 999,
        "full_name": "Test User",
        "image_urls": [
            # Updated image URLs
            "https://storage.googleapis.com/your-bucket/test-user-updated.jpg"
        ]
    }
}

response = requests.post(
    f"{FR_API_URL}/api/v1/embeddings/update",
    json=payload
)

print(f"Status Code: {response.status_code}")
print(f"Response: {json.dumps(response.json(), indent=2)}")

## Test 4: Delete User Embedding

In [ ]:
payload = {
    "action": "delete_user",
    "client_slug": CLIENT_SLUG,
    "user_data": {
        "id": 999,
        "full_name": "Test User"
    }
}

response = requests.post(
    f"{FR_API_URL}/api/v1/embeddings/update",
    json=payload
)

print(f"Status Code: {response.status_code}")
print(f"Response: {json.dumps(response.json(), indent=2)}")

## Test 5: Rebuild Database

In [ ]:
payload = {
    "client_slug": CLIENT_SLUG
}

response = requests.post(
    f"{FR_API_URL}/api/v1/embeddings/rebuild",
    json=payload
)

print(f"Status Code: {response.status_code}")
print(f"Response: {json.dumps(response.json(), indent=2)}")

## Test 6: End-to-End Test via Backend API

This test creates a real user via the backend API and verifies that the FR service is notified.

In [ ]:
# Step 1: Login to get token
login_payload = {
    "username": "your_username",  # Update with real credentials
    "password": "your_password",
    "client_slug": CLIENT_SLUG
}

login_response = requests.post(
    f"{BACKEND_URL}/api/auth/login",
    json=login_payload
)

if login_response.status_code == 200:
    token = login_response.json().get('token')
    print(f"✓ Logged in successfully")
    print(f"Token: {token[:20]}...")
else:
    print(f"✗ Login failed: {login_response.status_code}")
    print(login_response.text)

In [ ]:
# Step 2: Create a user with images
# Note: You'll need actual image files for this

from pathlib import Path

# Update with actual image path
image_path = Path("path/to/test/image.jpg")

if image_path.exists():
    files = {
        'images': open(image_path, 'rb')
    }
    
    data = {
        'full_name': 'E2E Test User',
        'status': 'out'
    }
    
    headers = {
        'Authorization': f'Bearer {token}'
    }
    
    response = requests.post(
        f"{BACKEND_URL}/api/users",
        data=data,
        files=files,
        headers=headers
    )
    
    print(f"Status Code: {response.status_code}")
    print(f"Response: {json.dumps(response.json(), indent=2)}")
    
    # Step 3: Check FR service health to verify embedding count increased
    health_response = requests.get(f"{FR_API_URL}/api/v1/health")
    print(f"\nFR Service Health After User Creation:")
    print(json.dumps(health_response.json(), indent=2))
else:
    print(f"✗ Image file not found: {image_path}")

## Test 7: Monitor Docker Logs

Run these commands in your terminal to monitor logs:

```bash
# Backend FR notifications
docker compose logs -f backend | grep -i 'facerecognition\|face_model_update'

# Face Recognition service logs
docker compose logs -f face-recognition

# Both together
docker compose logs -f backend face-recognition
```